# Portfolio Optimization Solver Comparison

This notebook provides comprehensive comparison capabilities for multiple portfolio optimization solvers.
It extends the basic trio_comparison function to handle an arbitrary number of solvers with advanced
statistical analysis and visualization capabilities.

## Supported Solvers:
- **Genetic Algorithm (GA)**: Evolutionary optimization with population-based search
- **Scipy Minimize (BF)**: Classical optimization using SLSQP method  
- **Riskfolio HRP**: Hierarchical Risk Parity using clustering
- **D-Wave Classical**: Various classical samplers (Simulated Annealing, Tabu, Steepest Descent)
- **D-Wave Quantum CQM**: Quantum Constrained Quadratic Model solvers

## Analysis Features:
- Performance comparison across multiple periods
- Statistical significance testing
- Risk-adjusted return metrics
- Execution time analysis
- Portfolio composition analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import os
import logging
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from scipy import stats
from typing import List, Dict, Tuple, Optional
import itertools

# Configuration
SEED = 12
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Utility Functions

In [ ]:
def read_pickle_dict(file_path: str) -> Optional[Dict]:
    """Safe pickle file reader with error handling."""
    try:
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        logger.warning(f"File not found: {file_path}")
        return None
    except Exception as e:
        logger.error(f"Error reading {file_path}: {e}")
        return None

def extract_solver_name(file_path: str) -> str:
    """Extract solver name from file path."""
    # Extract from pattern like '../results/results_10_{}_solver_name.pkl'
    basename = os.path.basename(file_path)
    # Remove .pkl and split by underscores
    parts = basename.replace('.pkl', '').split('_')
    # Return the last parts as solver name
    if len(parts) >= 3:
        return '_'.join(parts[3:])  # Everything after 'results_size_{}'
    return basename

def calculate_portfolio_metrics(returns: np.ndarray, benchmark_returns: np.ndarray = None) -> Dict:
    """Calculate comprehensive portfolio performance metrics."""
    returns_array = np.array(returns)
    
    metrics = {
        'total_return': returns_array[-1] if len(returns_array) > 0 else 0,
        'mean_return': np.mean(returns_array),
        'volatility': np.std(returns_array),
        'sharpe_ratio': np.mean(returns_array) / np.std(returns_array) if np.std(returns_array) > 0 else 0,
        'max_drawdown': np.min(returns_array) if len(returns_array) > 0 else 0,
        'num_periods': len(returns_array)
    }
    
    if benchmark_returns is not None:
        benchmark_array = np.array(benchmark_returns)
        if len(benchmark_array) == len(returns_array):
            excess_returns = returns_array - benchmark_array
            metrics['alpha'] = np.mean(excess_returns)
            
            # Beta calculation (simple correlation-based)
            if np.std(benchmark_array) > 0:
                metrics['beta'] = np.corrcoef(returns_array, benchmark_array)[0,1] * (np.std(returns_array) / np.std(benchmark_array))
            else:
                metrics['beta'] = 0
                
            # Information ratio
            if np.std(excess_returns) > 0:
                metrics['information_ratio'] = np.mean(excess_returns) / np.std(excess_returns)
            else:
                metrics['information_ratio'] = 0
    
    return metrics

## Multi-Solver Comparison Class

In [ ]:
class MultiSolverComparison:
    """Comprehensive comparison of multiple portfolio optimization solvers."""
    
    def __init__(self, results_dir: str = '../results'):
        self.results_dir = Path(results_dir)
        self.solvers_data = {}
        self.comparison_results = {}
        
    def load_solver_results(self, solver_paths: List[str], n_periods: int) -> Dict:
        """Load results for multiple solvers across all periods."""
        solvers_data = {}
        
        for solver_path_template in solver_paths:
            solver_name = extract_solver_name(solver_path_template)
            solver_results = []
            
            for period in range(n_periods):
                file_path = solver_path_template.format(period)
                result = read_pickle_dict(file_path)
                if result is not None:
                    solver_results.append(result)
                else:
                    logger.warning(f"Missing data for {solver_name} period {period}")
            
            if solver_results:
                solvers_data[solver_name] = solver_results
                logger.info(f"Loaded {len(solver_results)} periods for {solver_name}")
            else:
                logger.warning(f"No data found for {solver_name}")
        
        self.solvers_data = solvers_data
        return solvers_data
    
    def compare_performance(self, period_idx: int = 0, show_benchmark: bool = True) -> None:
        """Create performance comparison plot for a specific period."""
        if not self.solvers_data:
            logger.error("No solver data loaded. Call load_solver_results first.")
            return
        
        plt.figure(figsize=(15, 8))
        
        # Color palette for solvers
        colors = plt.cm.Set3(np.linspace(0, 1, len(self.solvers_data)))
        
        benchmark_plotted = False
        
        for i, (solver_name, results_list) in enumerate(self.solvers_data.items()):
            if period_idx >= len(results_list):
                logger.warning(f"Period {period_idx} not available for {solver_name}")
                continue
                
            result = results_list[period_idx]
            portfolio_returns = result.get('portfolio_cumulative_returns', [])
            benchmark_returns = result.get('benchmark_cumulative_returns', [])
            
            if portfolio_returns:
                plt.plot(portfolio_returns, label=f'{solver_name}', 
                        color=colors[i], linewidth=2, alpha=0.8)
            
            # Plot benchmark only once
            if show_benchmark and benchmark_returns and not benchmark_plotted:
                plt.plot(benchmark_returns, label='Benchmark (S&P 500)', 
                        color='black', linestyle='--', linewidth=2, alpha=0.7)
                benchmark_plotted = True
        
        plt.title(f'Portfolio Performance Comparison - Period {period_idx}', fontsize=16, fontweight='bold')
        plt.xlabel('Trading Days', fontsize=12)
        plt.ylabel('Cumulative Returns', fontsize=12)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def analyze_all_periods(self) -> pd.DataFrame:
        """Analyze performance across all periods and return summary statistics."""
        if not self.solvers_data:
            logger.error("No solver data loaded. Call load_solver_results first.")
            return pd.DataFrame()
        
        analysis_results = []
        
        for solver_name, results_list in self.solvers_data.items():
            solver_metrics = []
            execution_times = []
            
            for result in results_list:
                portfolio_returns = result.get('portfolio_cumulative_returns', [])
                benchmark_returns = result.get('benchmark_cumulative_returns', [])
                exec_time = result.get('total_run_time', 0)
                
                if portfolio_returns:
                    metrics = calculate_portfolio_metrics(portfolio_returns, benchmark_returns)
                    solver_metrics.append(metrics)
                    execution_times.append(exec_time)
            
            if solver_metrics:
                # Aggregate metrics across periods
                avg_metrics = {
                    'solver': solver_name,
                    'num_periods': len(solver_metrics),
                    'avg_total_return': np.mean([m['total_return'] for m in solver_metrics]),
                    'std_total_return': np.std([m['total_return'] for m in solver_metrics]),
                    'avg_sharpe_ratio': np.mean([m['sharpe_ratio'] for m in solver_metrics]),
                    'avg_volatility': np.mean([m['volatility'] for m in solver_metrics]),
                    'avg_max_drawdown': np.mean([m['max_drawdown'] for m in solver_metrics]),
                    'avg_execution_time': np.mean(execution_times),
                    'std_execution_time': np.std(execution_times)
                }
                
                # Add alpha and beta if available
                alpha_values = [m.get('alpha', np.nan) for m in solver_metrics]
                beta_values = [m.get('beta', np.nan) for m in solver_metrics]
                info_ratio_values = [m.get('information_ratio', np.nan) for m in solver_metrics]
                
                if not all(np.isnan(alpha_values)):
                    avg_metrics['avg_alpha'] = np.nanmean(alpha_values)
                    avg_metrics['avg_beta'] = np.nanmean(beta_values)
                    avg_metrics['avg_information_ratio'] = np.nanmean(info_ratio_values)
                
                analysis_results.append(avg_metrics)
        
        results_df = pd.DataFrame(analysis_results)
        if not results_df.empty:
            results_df = results_df.round(4)
        
        self.comparison_results = results_df
        return results_df
    
    def plot_performance_metrics(self) -> None:
        """Create comprehensive performance metrics visualization."""
        if self.comparison_results.empty:
            logger.error("No comparison results available. Run analyze_all_periods first.")
            return
        
        df = self.comparison_results
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Portfolio Optimization Solver Comparison', fontsize=16, fontweight='bold')
        
        # 1. Total Return
        ax1 = axes[0, 0]
        bars1 = ax1.bar(df['solver'], df['avg_total_return'], 
                       yerr=df['std_total_return'], capsize=5, alpha=0.8)
        ax1.set_title('Average Total Return')
        ax1.set_ylabel('Return')
        ax1.tick_params(axis='x', rotation=45)
        
        # 2. Sharpe Ratio
        ax2 = axes[0, 1]
        bars2 = ax2.bar(df['solver'], df['avg_sharpe_ratio'], alpha=0.8, color='orange')
        ax2.set_title('Average Sharpe Ratio')
        ax2.set_ylabel('Sharpe Ratio')
        ax2.tick_params(axis='x', rotation=45)
        
        # 3. Volatility
        ax3 = axes[0, 2]
        bars3 = ax3.bar(df['solver'], df['avg_volatility'], alpha=0.8, color='red')
        ax3.set_title('Average Volatility')
        ax3.set_ylabel('Volatility')
        ax3.tick_params(axis='x', rotation=45)
        
        # 4. Max Drawdown
        ax4 = axes[1, 0]
        bars4 = ax4.bar(df['solver'], df['avg_max_drawdown'], alpha=0.8, color='purple')
        ax4.set_title('Average Max Drawdown')
        ax4.set_ylabel('Max Drawdown')
        ax4.tick_params(axis='x', rotation=45)
        
        # 5. Execution Time
        ax5 = axes[1, 1]
        bars5 = ax5.bar(df['solver'], df['avg_execution_time'], 
                       yerr=df['std_execution_time'], capsize=5, alpha=0.8, color='green')
        ax5.set_title('Average Execution Time')
        ax5.set_ylabel('Time (seconds)')
        ax5.tick_params(axis='x', rotation=45)
        
        # 6. Risk-Return Scatter
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df['avg_volatility'], df['avg_total_return'], 
                             s=100, alpha=0.7, c=range(len(df)), cmap='viridis')
        ax6.set_title('Risk-Return Profile')
        ax6.set_xlabel('Volatility')
        ax6.set_ylabel('Total Return')
        
        # Add solver labels to scatter plot
        for i, solver in enumerate(df['solver']):
            ax6.annotate(solver, (df.iloc[i]['avg_volatility'], df.iloc[i]['avg_total_return']),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        plt.tight_layout()
        plt.show()
    
    def statistical_significance_test(self) -> pd.DataFrame:
        """Perform pairwise statistical significance tests between solvers."""
        if not self.solvers_data:
            logger.error("No solver data loaded. Call load_solver_results first.")
            return pd.DataFrame()
        
        # Extract returns for each solver
        solver_returns = {}
        for solver_name, results_list in self.solvers_data.items():
            returns = []
            for result in results_list:
                portfolio_returns = result.get('portfolio_cumulative_returns', [])
                if portfolio_returns:
                    returns.append(portfolio_returns[-1])  # Final return
            solver_returns[solver_name] = returns
        
        # Pairwise t-tests
        solver_names = list(solver_returns.keys())
        n_solvers = len(solver_names)
        
        p_values = np.ones((n_solvers, n_solvers))
        t_stats = np.zeros((n_solvers, n_solvers))
        
        for i in range(n_solvers):
            for j in range(i+1, n_solvers):
                solver1, solver2 = solver_names[i], solver_names[j]
                returns1, returns2 = solver_returns[solver1], solver_returns[solver2]
                
                if len(returns1) > 1 and len(returns2) > 1:
                    t_stat, p_val = stats.ttest_ind(returns1, returns2)
                    p_values[i, j] = p_values[j, i] = p_val
                    t_stats[i, j] = t_stat
                    t_stats[j, i] = -t_stat
        
        # Create result DataFrame
        p_values_df = pd.DataFrame(p_values, index=solver_names, columns=solver_names)
        t_stats_df = pd.DataFrame(t_stats, index=solver_names, columns=solver_names)
        
        return {'p_values': p_values_df, 't_statistics': t_stats_df}
    
    def plot_significance_heatmap(self, significance_results: Dict) -> None:
        """Plot heatmap of statistical significance between solvers."""
        if not significance_results:
            logger.error("No significance results provided")
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # P-values heatmap
        sns.heatmap(significance_results['p_values'], annot=True, cmap='RdYlBu_r', 
                   center=0.05, vmin=0, vmax=0.1, ax=ax1, cbar_kws={'label': 'p-value'})
        ax1.set_title('P-values Matrix\n(Red = Significant Difference)')
        
        # T-statistics heatmap
        sns.heatmap(significance_results['t_statistics'], annot=True, cmap='RdBu_r', 
                   center=0, ax=ax2, cbar_kws={'label': 't-statistic'})
        ax2.set_title('T-statistics Matrix\n(Blue = First solver better)')
        
        plt.tight_layout()
        plt.show()
    
    def generate_report(self, output_file: str = None) -> str:
        """Generate comprehensive comparison report."""
        if self.comparison_results.empty:
            self.analyze_all_periods()
        
        report = []
        report.append("# Portfolio Optimization Solver Comparison Report")
        report.append(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("\n## Summary Statistics")
        
        df = self.comparison_results
        
        # Best performing solver by different metrics
        best_return = df.loc[df['avg_total_return'].idxmax(), 'solver']
        best_sharpe = df.loc[df['avg_sharpe_ratio'].idxmax(), 'solver']
        fastest = df.loc[df['avg_execution_time'].idxmin(), 'solver']
        
        report.append(f"- **Highest Average Return**: {best_return} ({df['avg_total_return'].max():.4f})")
        report.append(f"- **Best Sharpe Ratio**: {best_sharpe} ({df['avg_sharpe_ratio'].max():.4f})")
        report.append(f"- **Fastest Execution**: {fastest} ({df['avg_execution_time'].min():.2f}s)")
        
        report.append("\n## Detailed Results")
        report.append(df.to_string())
        
        # Statistical significance
        sig_results = self.statistical_significance_test()
        if sig_results:
            report.append("\n## Statistical Significance (p-values)")
            report.append(sig_results['p_values'].to_string())
        
        report_text = "\n".join(report)
        
        if output_file:
            with open(output_file, 'w') as f:
                f.write(report_text)
            logger.info(f"Report saved to {output_file}")
        
        return report_text

## Usage Examples

In [ ]:
# Initialize the comparison class
comparator = MultiSolverComparison()

# Example: Compare different solver types on 10 stocks
solver_paths_10 = [
    '../results/results_10_{}_ga.pkl',           # Genetic Algorithm
    '../results/results_10_{}_bf.pkl',           # Scipy Brute Force
    '../results/results_10_{}_hrp.pkl',          # Hierarchical Risk Parity
    '../results/results_10_{}_dwave_sa.pkl',     # D-Wave Simulated Annealing
    '../results/results_10_{}_dwave_tabu.pkl',   # D-Wave Tabu Search
    '../results/results_10_{}_dwave_greedy.pkl'  # D-Wave Steepest Descent
]

print("=" * 60)
print("PORTFOLIO OPTIMIZATION SOLVER COMPARISON")
print("Dataset: 10 Stocks")
print("=" * 60)

In [ ]:
# Load solver results
n_periods = 10  # Adjust based on your data
solver_data = comparator.load_solver_results(solver_paths_10, n_periods)

print(f"\nLoaded data for {len(solver_data)} solvers:")
for solver, data in solver_data.items():
    print(f"- {solver}: {len(data)} periods")

In [ ]:
# Compare performance for first few periods
for period in range(min(3, n_periods)):  # Show first 3 periods
    comparator.compare_performance(period_idx=period, show_benchmark=True)

In [ ]:
# Analyze all periods and show summary
results_df = comparator.analyze_all_periods()
print("\nSummary Statistics Across All Periods:")
print("=" * 50)
display(results_df)

In [ ]:
# Plot comprehensive performance metrics
comparator.plot_performance_metrics()

In [ ]:
# Statistical significance testing
significance_results = comparator.statistical_significance_test()
if significance_results:
    print("\nStatistical Significance Analysis:")
    print("P-values matrix (values < 0.05 indicate significant difference):")
    display(significance_results['p_values'])
    
    # Plot significance heatmap
    comparator.plot_significance_heatmap(significance_results)

In [ ]:
# Generate comprehensive report
report = comparator.generate_report('../results/solver_comparison_report_10_stocks.md')
print("\nGenerated Report:")
print("=" * 20)
print(report[:1000] + "..." if len(report) > 1000 else report)

## Additional Analysis Functions

In [ ]:
def compare_dataset_sizes():
    """Compare solver performance across different dataset sizes."""
    dataset_configs = {
        '10_stocks': '../results/results_10_{}_',
        '100_stocks': '../results/results_100_{}_',
        'full_dataset': '../results/results_full_{}_',
        'wide_dataset': '../results/results_wide_{}_'
    }
    
    solver_types = ['ga', 'bf', 'hrp', 'dwave_sa', 'dwave_tabu', 'dwave_greedy']
    
    comparison_results = {}
    
    for dataset_name, path_template in dataset_configs.items():
        print(f"\nAnalyzing {dataset_name}:")
        print("=" * 30)
        
        # Build paths for this dataset
        dataset_paths = [f"{path_template}{solver_type}.pkl" for solver_type in solver_types]
        
        # Create comparator for this dataset
        dataset_comparator = MultiSolverComparison()
        solver_data = dataset_comparator.load_solver_results(dataset_paths, n_periods)
        
        if solver_data:
            results_df = dataset_comparator.analyze_all_periods()
            comparison_results[dataset_name] = results_df
            
            print(f"Loaded {len(solver_data)} solvers")
            if not results_df.empty:
                best_return = results_df.loc[results_df['avg_total_return'].idxmax(), 'solver']
                best_sharpe = results_df.loc[results_df['avg_sharpe_ratio'].idxmax(), 'solver']
                print(f"Best Return: {best_return}")
                print(f"Best Sharpe: {best_sharpe}")
        else:
            print("No data found for this dataset")
    
    return comparison_results

def plot_scalability_analysis(comparison_results: Dict):
    """Plot how solver performance scales with dataset size."""
    if not comparison_results:
        print("No comparison results to plot")
        return
    
    # Extract data for plotting
    all_solvers = set()
    for df in comparison_results.values():
        if not df.empty:
            all_solvers.update(df['solver'].tolist())
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Solver Scalability Analysis', fontsize=16, fontweight='bold')
    
    metrics = ['avg_total_return', 'avg_sharpe_ratio', 'avg_volatility', 'avg_execution_time']
    titles = ['Average Total Return', 'Average Sharpe Ratio', 'Average Volatility', 'Average Execution Time']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 2, idx % 2]
        
        for solver in all_solvers:
            x_vals = []
            y_vals = []
            
            for dataset_name, df in comparison_results.items():
                if not df.empty and solver in df['solver'].values:
                    solver_row = df[df['solver'] == solver].iloc[0]
                    if metric in solver_row and not pd.isna(solver_row[metric]):
                        x_vals.append(dataset_name)
                        y_vals.append(solver_row[metric])
            
            if x_vals and y_vals:
                ax.plot(x_vals, y_vals, marker='o', label=solver, linewidth=2)
        
        ax.set_title(title)
        ax.set_xlabel('Dataset Size')
        ax.set_ylabel(metric.replace('avg_', '').replace('_', ' ').title())
        ax.tick_params(axis='x', rotation=45)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run scalability analysis
print("Running Scalability Analysis...")
print("This may take a while as it loads all datasets")
print("=" * 50)

In [ ]:
# Uncomment to run scalability analysis
# scalability_results = compare_dataset_sizes()
# plot_scalability_analysis(scalability_results)

## Custom Analysis Templates

Use these templates to perform specific comparisons:

In [ ]:
# Template 1: Compare only D-Wave solvers
def compare_dwave_solvers(dataset_size='10'):
    """Compare different D-Wave classical samplers."""
    dwave_paths = [
        f'../results/results_{dataset_size}_{{}}_dwave_sa.pkl',
        f'../results/results_{dataset_size}_{{}}_dwave_tabu.pkl',
        f'../results/results_{dataset_size}_{{}}_dwave_greedy.pkl'
    ]
    
    comparator = MultiSolverComparison()
    solver_data = comparator.load_solver_results(dwave_paths, n_periods)
    
    if solver_data:
        comparator.compare_performance(period_idx=0)
        results_df = comparator.analyze_all_periods()
        display(results_df)
        comparator.plot_performance_metrics()
        
        return results_df
    else:
        print(f"No D-Wave results found for dataset size {dataset_size}")
        return None

# Template 2: Compare traditional vs quantum-inspired solvers
def compare_traditional_vs_quantum():
    """Compare traditional optimization vs quantum-inspired methods."""
    traditional_paths = [
        '../results/results_10_{}_ga.pkl',
        '../results/results_10_{}_bf.pkl',
        '../results/results_10_{}_hrp.pkl'
    ]
    
    quantum_paths = [
        '../results/results_10_{}_dwave_sa.pkl',
        '../results/results_10_{}_dwave_tabu.pkl',
        '../results/results_10_{}_dwave_greedy.pkl'
    ]
    
    all_paths = traditional_paths + quantum_paths
    
    comparator = MultiSolverComparison()
    solver_data = comparator.load_solver_results(all_paths, n_periods)
    
    if solver_data:
        results_df = comparator.analyze_all_periods()
        
        # Separate results
        traditional_solvers = ['ga', 'bf', 'hrp']
        quantum_solvers = ['dwave_sa', 'dwave_tabu', 'dwave_greedy']
        
        trad_results = results_df[results_df['solver'].isin(traditional_solvers)]
        quantum_results = results_df[results_df['solver'].isin(quantum_solvers)]
        
        print("Traditional Methods:")
        display(trad_results)
        print("\nQuantum-Inspired Methods:")
        display(quantum_results)
        
        return {'traditional': trad_results, 'quantum': quantum_results}
    
    return None

print("Analysis templates loaded. Use the functions above for specific comparisons.")

## Conclusion

This notebook provides comprehensive tools for comparing portfolio optimization solvers:

1. **Multi-solver comparison** with arbitrary number of algorithms
2. **Statistical analysis** including significance testing
3. **Performance visualization** across multiple metrics
4. **Scalability analysis** across different dataset sizes
5. **Automated reporting** with detailed results

The `MultiSolverComparison` class can be extended to include additional metrics, visualization types, or analysis methods as needed.